# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'product page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'social media - LinkedIn',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media - Twitter',
   'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media - Facebook',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Found 14 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
7 days ago
•
337k
•
1.81k
CohereLabs/cohere-transcribe-03-2026
Updated
1 day ago
•
50.5k
•
594
mistralai/Voxtral-4B-TTS-2603
Updated
4 days ago
•
3.72k
•
544
baidu/Qianfan-OCR
Updated
5 days ago
•
17.6k
•
691
chromadb/context-1
Updated
1 day ago
•
2.39k
•
306
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.72k
Wan2.2 14B Preview
🐌
1.72k
generate a video from an image with a text prompt
Running
on
Zero
Featured
322
DLSS 5 Anything
🎮
322
Turn any image into a DLSS 5 meme (using FLUX.2-klein-9b-kv)
Running
360
Free Unlimited Google Veo 3
🌖
360


In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n7 days ago\n•\n337k\n•\n1.81k\nCohereLabs/cohere-transcribe-03-2026\nUpdated\n1 day ago\n•\n50.5k\n•\n594\nmistralai/Voxtral-4B-TTS-2603\nUpdated\n4 days ago\n•\n3.72k\n•\n544\nbaidu/Qianfan-OCR\nUpdated\n5 days ago\n•\n17.6k\n•\n691\nchromadb/context-1\nUpdated\n1 day ago\n•\n2.39k\n•\n306\nBrowse 2M+ models\nSpaces\nRun

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI community dedicated to building the future of machine learning through collaboration, openness, and innovation. It serves as the central platform where the global machine learning community comes together to create, discover, and share models, datasets, and applications across all modalities—text, image, video, audio, and even 3D.

Founded on the ideals of open source and ethical AI, Hugging Face empowers engineers, scientists, and end-users alike to accelerate machine learning (ML) development by providing an accessible and collaborative ecosystem.

---

## What We Offer

- **2 Million+ Models:** Explore and contribute to an extensive repository of AI models updated regularly, covering a vast range of ML use cases.
- **500k+ Datasets:** Access diverse datasets curated by the community for training and benchmarking AI models.
- **1 Million+ Applications:** Discover and deploy AI-powered applications running on our platform.
- **Spaces & Buckets:** Host and showcase machine learning applications and projects with ease.
- **Open Source Stack:** Build faster and more efficiently using Hugging Face’s open-source tools and APIs across various modalities.
- **Enterprise Solutions:** Tailored AI services for businesses to integrate cutting-edge technology securely and efficiently.
- **Paid Compute:** Accelerate your ML workflows with scalable and reliable computing resources provided by Hugging Face.

---

## Our Community & Customers

Hugging Face brings together a vibrant, fast-growing community that includes:

- Machine learning engineers and scientists pushing the boundaries of AI.
- Researchers and academics contributing datasets and models to the open-source knowledge base.
- Developers creating innovative AI applications accessible worldwide.
- Enterprises seeking AI-driven solutions and expertise.
- Enthusiasts and learners building their portfolios and profiles in the ML field.

The platform fosters collaboration that advances AI development in an open, ethical, and inclusive manner.

---

## Company Culture

At Hugging Face, we believe in:

- **Open Collaboration:** Everyone is encouraged to share and build upon each other’s work to fuel innovation.
- **Ethical AI:** Commitment to developing AI technologies responsibly and transparently.
- **Community Empowerment:** Enabling the next generation of AI professionals through accessible tools and resources.
- **Diversity and Inclusion:** Building a welcoming environment for all contributors regardless of background.
- **Continuous Learning:** Supporting growth and portfolio building in machine learning and AI.

---

## Careers

Join Hugging Face and be part of an exciting journey to shape the future of AI! Our team is composed of talented individuals passionate about democratizing machine learning and driving its positive impact globally.

- **Opportunities:** Engineering, research, product, business development, community management, and more.
- **Growth:** Work with cutting-edge technology and a world-class community.
- **Impact:** Contribute to open-source projects and enterprise solutions used worldwide.
- **Culture:** Collaborative, inclusive, transparent, and innovation-driven.

To explore current openings and apply, please visit our website at [huggingface.co](https://huggingface.co).

---

## Get Involved

Whether you want to explore AI models, contribute a dataset, build an application, or leverage enterprise solutions, Hugging Face is your home for machine learning collaboration.

- Sign up and start exploring at: [huggingface.co](https://huggingface.co)
- Join the conversations, share your work, and accelerate your ML journey today!

---

**Hugging Face — The AI community building the future.**  
Empowering innovation through open collaboration, for a better, ethical AI tomorrow.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future. It is the premier platform where the global machine learning (ML) community collaborates on models, datasets, and applications across multiple modalities including text, image, video, audio, and even 3D.

With over **2 million models**, **500,000 datasets**, and **1 million+ applications**, Hugging Face offers an open and collaborative environment for researchers, developers, and enterprises to create, discover, and share cutting-edge AI and ML technologies.

---

## What Hugging Face Offers

### Core Platform Features:
- **Models Hub:** Host and browse over 2 million public ML models covering a variety of tasks and domains.
- **Datasets:** Access more than 500,000 public datasets curated for various AI tasks.
- **Spaces:** Create and share AI-powered applications using web-hosted demos.
- **Buckets & Private Storage:** Manage and store datasets or models privately with organization control.
- **Collaboration Tools:** Build your ML portfolio by sharing your work with the community.

### Enterprise & Team Plans:
Hugging Face offers flexible subscription plans designed to scale with your organization's needs:

- **Team Plan ($20/user/month):** 
  - Single Sign-On (SSO) integration for secure access
  - Granular access control with resource groups
  - Audit logs for full visibility
  - Increased private storage (1 TB per member)
  - Advanced compute like ZeroGPU with quota boost
  - Priority customer support
  - Dataset viewer for collaboration on private data

- **Enterprise Plan:** 
  - Custom contracts with advanced security policies
  - Data regionalization and compliance management
  - Dedicated support and billing management
  - Enhanced scalability and organization-wide controls

---

## Pricing Highlights

- **PRO Account ($9/month):** For individual professionals who want enhanced capabilities such as:
  - 10× private storage capacity
  - 2× public storage capacity
  - 20× inference credits included
  - Higher priority on compute queues (ZeroGPU)
  - Ability to publish blog posts and showcase work with badges

- **Team Accounts ($20/user/month):** Designed for growing teams requiring collaboration and administrative control.

---

## Company Culture & Community

Hugging Face fosters a **collaborative, open-source-driven community culture** that empowers machine learning practitioners to accelerate AI research and deployment. The platform embodies:

- **Open collaboration:** Unlimited hosting of public models, datasets, and applications.
- **Inclusivity:** Support for all AI modalities and diverse use cases.
- **Skill-building:** Tools and opportunities for users to showcase and improve their ML skills.
- **Innovation:** Encouragement to explore state-of-the-art models, including trending large language models and multimodal AI systems.

The company’s philosophy encourages users to build portfolios, share ideas, and contribute back to the AI ecosystem, making it a magnet for researchers, developers, hobbyists, and enterprise users alike.

---

## Customers & Use Cases

Hugging Face serves a wide range of users and organizations:
- **AI researchers and data scientists** using pre-trained or custom models.
- **Startups and developers** building AI-powered applications across industries like gaming, healthcare, finance, and more.
- **Large enterprises** adopting robust AI infrastructure with enterprise-grade security and collaboration.
- **AI educators and students** for learning and experimentation.

Trending models and datasets on the platform include advanced text-to-speech systems, OCR, video generation from images, and reasoning-optimized language models, demonstrating its versatility.

---

## Careers & Opportunities

Hugging Face offers exciting career prospects in a fast-growing field at the forefront of AI innovation. Team members thrive in an environment that values:

- **Cutting-edge AI research and engineering**
- **Cross-functional collaboration**
- **Open source contributions**
- **Customer-centric product development**

Prospective recruits can look forward to contributing to tools used globally by millions in the AI community, with opportunities for growth and making a tangible impact on the future of machine learning.

---

## Join Hugging Face

Whether you want to accelerate your personal AI projects, collaborate with teammates, or scale AI across your enterprise, Hugging Face offers the tools, community, and support to help you succeed.

- **Explore over 2 million models and hundreds of thousands of datasets today.**
- **Create your portfolio and share your AI innovations with the world.**
- **Leverage team and enterprise plans for advanced security, access control, and compute power.**

**Start building the future of AI with Hugging Face.**

[Sign Up at Hugging Face](https://huggingface.co)

---

*Hugging Face – The AI community building the future.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>